## Backstage als Container-Image vorbereiten

Minimaler Ablauf für eine lokale `mybackstage`-Installation ohne Kubernetes.

### Produktionskonfiguration erstellen



In [ ]:
%%bash
cd ~/mybackstage
rm app-config.*.yaml
cat > app-config.production.yaml <<'EOF'
app:
  title: ${BACKSTAGE_NAME}
  baseUrl: http://${BACKSTAGE_HOSTNAME}:${BACKSTAGE_PORT}
  listen:
    host: 0.0.0.0   # Optional: Auf allen Interfaces lauschen, falls nötig

  # Enable all packages by default, this will discover packages from packages/app/package.json
  packages: all

  extensions:
    # Configure the catalog index page to be the root page, this is normally mounted on /catalog
    - page:catalog:
        config:
          path: /

organization:
  name: ${BACKSTAGE_NAME}

backend:
  baseUrl:  http://${BACKSTAGE_HOSTNAME}:${BACKSTAGE_PORT}
  listen:
    port: 7007
    host: 0.0.0.0   
  auth:
    keys:
      - secret: "change-me-to-a-long-random-secret-change-this"    
  csp:
    connect-src: ["'self'", 'http:', 'https:']
    upgrade-insecure-requests: false    
  cors:
    origin: http://${BACKSTAGE_HOSTNAME}:${BACKSTAGE_PORT}
    methods: [GET, HEAD, PATCH, POST, PUT, DELETE]
    credentials: true
  # This is for local development only, it is not recommended to use this in production
  # The production database configuration is stored in app-config.production.yaml
  database:
    client: better-sqlite3
    connection: ':memory:'
  # workingDirectory: /tmp # Use this to configure a working directory for the scaffolder, defaults to the OS temp-dir
  # see https://backstage.io/docs/ai/mcp-actions#actions-configuration for more details
  actions:
    pluginSources:
      - auth
      - catalog
      - scaffolder
  # workingDirectory: /tmp # Use this to configure a working directory for the scaffolder, defaults to the OS temp-dir

integrations:
  github:
    - host: github.com
      token: ${GITHUB_TOKEN}
  gitlab:
    - host: gitlab.com
      token: ${GITLAB_TOKEN}

proxy:
  ### Example for how to add a proxy endpoint for the frontend.
  ### A typical reason to do this is to handle HTTPS and CORS for internal services.
  # endpoints:
  #   '/test':
  #     target: 'https://example.com'
  #     changeOrigin: true

# Reference documentation http://backstage.io/docs/features/techdocs/configuration
# Note: After experimenting with basic setup, use CI/CD to generate docs
# and an external cloud storage when deploying TechDocs for production use-case.
# https://backstage.io/docs/features/techdocs/how-to-guides#how-to-migrate-from-techdocs-basic-to-recommended-deployment-approach
techdocs:
  builder: 'local' # Alternatives - 'external'
  generator:
    runIn: 'docker' # Alternatives - 'local'
  publisher:
    type: 'local' # Alternatives - 'googleGcs' or 'awsS3'. Read documentation for using alternatives.

# in Produktiven Umgebungen entfernen!
auth:
  providers:
    guest:
      dangerouslyAllowOutsideDevelopment: true

  # see https://backstage.io/docs/ai/mcp-actions#client-id-metadata-documents
  # to learn more about client id metadata documents
  experimentalClientIdMetadataDocuments:
    enabled: false

scaffolder:
  # see https://backstage.io/docs/features/software-templates/configuration for software template options

catalog:
  import:
    entityFilename: catalog-info.yaml
    pullRequestBranchName: backstage-integration
  rules:
    - allow: [User, Group, Component, System, API, Resource, Location, Template]
  locations:
    # Auto Shop GmbH
    - type: url
      target: https://gitlab.com/ch-mc-b/autoshop-ms/infra/backstage/-/blob/main/org.yaml
      rules:
        - allow: [Component, System, API, Resource, Location, Templates]

kubernetes:
  # see https://backstage.io/docs/features/kubernetes/configuration for kubernetes configuration options

# see https://backstage.io/docs/permissions/getting-started for more on the permission framework
permission:
  # setting this to `false` will disable permissions
  enabled: false

# see https://backstage.io/docs/ai/mcp-actions for more details
mcpActions:
  name: 'My Company Backstage' # defaults to "backstage"
  description: 'Tools for managing your software catalog, creating new services from templates, and exploring your developer portal' # optional
EOF


### Backstage für Produktion bauen

Zuerst werden die Abhängigkeiten installiert, danach die Typen und das Backend-Bundle erzeugt. Das Frontend wird dabei in das Backend integriert.

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage
yarn install --immutable
yarn tsc
yarn build:backend

## Container-Image erstellen

Das Dockerfile liegt im Backend-Paket. Der Build-Kontext bleibt jedoch das Stammverzeichnis des Backstage-Projekts.

In [ ]:
%%bash
cd ~/mybackstage
docker image build --file packages/backend/Dockerfile --tag mybackstage:1.0.0 .

### Container starten

Die Umgebungsvariablen werden aus `~/data/env-platen.py` geladen. Backstage ist danach über Port `7007` erreichbar.

In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME="$(cat ~/data/server-ip)"
export BACKSTAGE_NAME="Backstage Produktion"
export BACKSTAGE_PORT="7007"

echo "http://$(cat ~/data/server-ip):${BACKSTAGE_PORT}"
docker run \
  --name mybackstage \
  --rm \
  --env BACKSTAGE_HOSTNAME \
  --env BACKSTAGE_NAME \
  --env BACKSTAGE_PORT \
  --env GITHUB_TOKEN \
  --env GITLAB_TOKEN \
  --publish 7007:7007 \
  mybackstage:1.0.0